In [ ]:
"""
CER – Renewable Energy Community · ML Analysis
===============================================
Bachelor's Thesis – Computer Engineering
Google Colab

Input (upload to Colab):
    cer_consumi_orari.csv   →  timestamp ; member_id ; consumo_kwh
    cer_members.csv         →  member_id , tipo , kwp_fv

Output: full dataset, forecasts, charts A-F, economic impact
"""

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import zipfile, os, warnings, base64
from sklearn.ensemble      import RandomForestRegressor
from sklearn.metrics       import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from IPython.display       import display, HTML

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white",
                     "axes.grid":True,"grid.alpha":0.3,"font.size":11})
SEED = 42
np.random.seed(SEED)

COLORS = {"reale":"#185FA5","previsto":"#D85A30","fv":"#BA7517",
          "shared":"#0F6E56","neutral":"#888780"}
MESI   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

print("  CER – Renewable Energy Community")
print("  Input: cer_consumi_orari.csv + cer_members.csv")


# ══════════════════════════════════════════════════════════════════════════════
# PART 1 – DATA LOADING AND PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

print("\n▶ PART 1 – Data loading and preprocessing")

# Load input files
df = pd.read_csv("cer_consumi_orari.csv", sep=";", decimal=",",
                 parse_dates=["timestamp"])
members = pd.read_csv("cer_members.csv")

df = df.sort_values(["member_id","timestamp"]).reset_index(drop=True)
df = df.merge(members, on="member_id", how="left")

print(f"  Rows:    {len(df):,}")
print(f"  Users:   {df['member_id'].nunique()}")
print(f"  Period:  {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

# Replace NaN values with 0
df["consumo_kwh"] = df["consumo_kwh"].fillna(0)

# Temporal features
df["ora"]              = df["timestamp"].dt.hour
df["giorno_settimana"] = df["timestamp"].dt.dayofweek
df["mese"]             = df["timestamp"].dt.month
df["is_weekend"]       = (df["giorno_settimana"] >= 5).astype(int)

# Italian public holidays 2023
festivita = pd.to_datetime([
    "2023-01-01","2023-01-06","2023-04-09","2023-04-10","2023-04-25",
    "2023-05-01","2023-06-02","2023-08-15","2023-11-01",
    "2023-12-08","2023-12-25","2023-12-26",
]).normalize()
df["is_festivo"] = np.isin(df["timestamp"].dt.normalize(),
                            festivita).astype(int)

# Cyclic encoding – preserves periodicity for the ML model
df["ora_sin"]     = np.sin(2*np.pi*df["ora"]              /24)
df["ora_cos"]     = np.cos(2*np.pi*df["ora"]              /24)
df["giorno_sin"]  = np.sin(2*np.pi*df["giorno_settimana"] /7)
df["giorno_cos"]  = np.cos(2*np.pi*df["giorno_settimana"] /7)
df["mese_sin"]    = np.sin(2*np.pi*df["mese"]             /12)
df["mese_cos"]    = np.cos(2*np.pi*df["mese"]             /12)

# PV production from PVGIS API (Benevento 41.13°N, 14.78°E)
print("  Fetching PVGIS data ...")

def get_pvgis(kwp, tilt=30, azimuth=0):
    idx = pd.date_range("2023-01-01", periods=8760, freq="h")
    params = {"lat":41.13,"lon":14.78,"peakpower":kwp,
              "pvtechchoice":"crystSi","mountingplace":"building",
              "angle":tilt,"aspect":azimuth,"loss":14,
              "pvcalculation":1,"outputformat":"json",
              "startyear":2020,"endyear":2020}
    try:
        r = requests.get("https://re.jrc.ec.europa.eu/api/v5_2/seriescalc",
                         params=params, timeout=60)
        r.raise_for_status()
        dfp = pd.DataFrame(r.json()["outputs"]["hourly"])
        dfp["dt"]  = pd.to_datetime(dfp["time"], format="%Y%m%d:%H%M")
        dfp["m"]   = dfp["dt"].dt.month
        dfp["h"]   = dfp["dt"].dt.hour
        media = dfp.groupby(["m","h"])["P"].mean() / 1000.0
        return np.array([media.get((m,h),0.0)
                         for m,h in zip(idx.month,idx.hour)])
    except Exception as e:
        print(f"    ✗ {e}"); return np.zeros(8760)

pv_cfg = {
    "M01":{"kwp":6.0, "tilt":30,"azimuth":  0},
    "M02":{"kwp":6.0, "tilt":28,"azimuth":-10},
    "M03":{"kwp":10.0,"tilt":30,"azimuth": 10},
}

df["produzione_fv_kwh"] = 0.0
for mid, cfg in pv_cfg.items():
    print(f"  {mid} ({cfg['kwp']:.0f} kWp) ...", end=" ", flush=True)
    serie = get_pvgis(**cfg)
    idx_u = df[df["member_id"]==mid].sort_values("timestamp").index
    df.loc[idx_u,"produzione_fv_kwh"] = serie
    print(f"✓ {serie.sum():.0f} kWh/year")

# Energy flows
df["autoconsumo_kwh"]     = np.minimum(df["consumo_kwh"],
                                        df["produzione_fv_kwh"])
df["energia_immessa_kwh"] = np.maximum(df["produzione_fv_kwh"]
                                        - df["consumo_kwh"], 0)
df["consumo_residuo_kwh"] = np.maximum(df["consumo_kwh"]
                                        - df["produzione_fv_kwh"], 0)

# Lag features – historical memory of consumption
for n, nome in [(1,"lag1"),(24,"lag24"),(168,"lag168")]:
    df[f"consumo_{nome}_kwh"] = (
        df.groupby("member_id")["consumo_kwh"]
          .transform(lambda x, n=n: x.shift(n).bfill())
    )

# CER-level aggregation
agg = df.groupby("timestamp").agg(
    consumo_CER_kwh         =("consumo_kwh",          "sum"),
    produzione_CER_kwh      =("produzione_fv_kwh",    "sum"),
    autoconsumo_CER_kwh     =("autoconsumo_kwh",      "sum"),
    energia_disponibile_kwh =("energia_immessa_kwh",  "sum"),
    consumo_residuo_CER_kwh =("consumo_residuo_kwh",  "sum"),
).reset_index()
agg["energia_condivisa_kwh"] = np.minimum(
    agg["consumo_residuo_CER_kwh"], agg["energia_disponibile_kwh"])

tc = agg["consumo_CER_kwh"].sum()
tf = agg["produzione_CER_kwh"].sum()
ts = agg["energia_condivisa_kwh"].sum()
ta = agg["autoconsumo_CER_kwh"].sum()

print(f"\n Performance Indices ")
print(f"  Total CER consumption:       {tc:>10,.0f} kWh/year")
print(f"  Total PV production:         {tf:>10,.0f} kWh/year")
print(f"  Individual self-consumption: {ta:>10,.0f} kWh/year")
print(f"  Shared energy:               {ts:>10,.0f} kWh/year")
print(f"  Self-consumption rate:       {100*ta/tf:>9.1f} %")
print(f"  Shared energy / PV rate:     {100*ts/tf:>9.1f} %")
print(f"  Consumption coverage rate:   {100*ts/tc:>9.1f} %")

# Save full dataset and CER aggregates
COLS = ["timestamp","member_id","tipo","kwp_fv",
        "ora","giorno_settimana","mese","is_weekend","is_festivo",
        "ora_sin","ora_cos","giorno_sin","giorno_cos","mese_sin","mese_cos",
        "consumo_kwh","produzione_fv_kwh",
        "autoconsumo_kwh","energia_immessa_kwh","consumo_residuo_kwh",
        "consumo_lag1_kwh","consumo_lag24_kwh","consumo_lag168_kwh"]
df[COLS].to_csv("cer_dataset_completo.csv", index=False)
agg.to_csv("cer_energia_condivisa.csv", index=False)
print("\n  ✓ CSV files saved")


# ══════════════════════════════════════════════════════════════════════════════
# PART 2 – MODEL TRAINING AND EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n▶ PART 2 – Random Forest Training")

le = LabelEncoder()
df["tipo_enc"] = le.fit_transform(df["tipo"])

FEATURES = ["mese","mese_sin","mese_cos","giorno_settimana","giorno_sin","giorno_cos",
            "ora","ora_sin","ora_cos","is_weekend","is_festivo",
            "tipo_enc","produzione_fv_kwh",
            "consumo_lag1_kwh","consumo_lag24_kwh","consumo_lag168_kwh"]
TARGET = "consumo_kwh"

# Temporal train/test split: Jan–Aug for training, Sep–Dec for testing
train = df[df["mese"] < 9].copy()
test  = df[df["mese"] >= 9].copy()
X_train,y_train = train[FEATURES], train[TARGET]
X_test, y_test  = test[FEATURES],  test[TARGET]

print(f"  Train: Jan–Aug  {len(train):,} samples  ({100*len(train)/len(df):.0f}%)")
print(f"  Test:  Sep–Dec  {len(test):,} samples  ({100*len(test)/len(df):.0f}%)")

rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_leaf=4, n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
test = test.copy()
test["consumo_previsto_kwh"] = y_pred

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred)/(y_test + 1e-6))) * 100

print(f"\n Global Metrics ")
print(f"  MAE  = {mae:.4f} kWh  |  RMSE = {rmse:.4f} kWh")
print(f"  MAPE = {mape:.2f}%     |  R²   = {r2:.4f}")

print("\n  By user type:")
for tipo in sorted(test["tipo"].unique()):
    m = test["tipo"]==tipo
    print(f"  {tipo:<15}  MAE={mean_absolute_error(y_test[m],y_pred[m]):.4f}"
          f"  R²={r2_score(y_test[m],y_pred[m]):.4f}")

print("\n  By time slot:")
for nome, ore in [("Night (0–5)",range(0,6)),("Morning (6–11)",range(6,12)),
                  ("Afternoon (12–17)",range(12,18)),("Evening (18–23)",range(18,24))]:
    m = test["ora"].isin(ore)
    print(f"  {nome:<22}  MAE={mean_absolute_error(y_test[m],y_pred[m]):.4f}"
          f"  R²={r2_score(y_test[m],y_pred[m]):.4f}")

test[["timestamp","member_id","tipo","mese","giorno_settimana","ora",
      "consumo_kwh","consumo_previsto_kwh","produzione_fv_kwh",
      "energia_immessa_kwh","consumo_residuo_kwh"]]\
    .to_csv("cer_previsioni.csv", index=False)


# ══════════════════════════════════════════════════════════════════════════════
# PART 3 – CHARTS
# ══════════════════════════════════════════════════════════════════════════════

print("\n▶ PART 3 – Charts")
GG = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

# ── Chart A: actual vs predicted · M01 · Sun–Tue ─────────────────────────────
fig, axes = plt.subplots(2,1,figsize=(10,8))
fig.suptitle("A  –  Actual vs predicted consumption · M01 · Sun–Tue",
             fontsize=13, fontweight="bold")

for ax, mese_sel, mese_label in zip(axes,
        [10, 12],
        ["October 2023 (test)", "December 2023 (test)"]):
    c = test[(test["member_id"]=="M01") & (test["mese"]==mese_sel)]\
        .sort_values("timestamp").reset_index(drop=True)
    # Find the first Sunday (dayofweek == 6)
    dom_idx = c[c["giorno_settimana"]==6].index[0]
    c3 = c.iloc[dom_idx : dom_idx+72].reset_index(drop=True)  # 3 days = 72 h
    x = range(len(c3))
    ax.plot(x, c3["consumo_kwh"],         color=COLORS["reale"],   lw=1.8, label="Actual")
    ax.plot(x, c3["consumo_previsto_kwh"], color=COLORS["previsto"],lw=1.4,
            ls="--", label="Predicted")
    ax.fill_between(x, c3["consumo_kwh"], c3["consumo_previsto_kwh"],
                    alpha=0.15, color=COLORS["previsto"])
    ax.set_xticks([d*24+12 for d in range(3)])
    ax.set_xticklabels(["Sun","Mon","Tue"], fontsize=10)
    for d in range(1,3): ax.axvline(d*24, color="gray", lw=0.5, alpha=0.4)
    ax.set_ylabel("kWh")
    ax.set_title(mese_label)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("grafico_A_reale_vs_previsto.png", dpi=150, bbox_inches="tight")
plt.show(); print("  ✓ Chart A")

# ── Chart B: scatter plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.suptitle("B  –  Actual vs predicted scatter",fontsize=13,fontweight="bold")
for ax, tipo in zip(axes,["prosumer_res","consumer_res","commercial"]):
    m=test["tipo"]==tipo; yr=y_test[m].values; yp=y_pred[m]
    lim=max(yr.max(),yp.max())*1.05
    ax.scatter(yr,yp,alpha=0.25,s=8,color=COLORS["reale"])
    ax.plot([0,lim],[0,lim],"k--",lw=1)
    ax.set_title(f"{tipo}\nR²={r2_score(yr,yp):.3f}")
    ax.set_xlabel("Actual (kWh)"); ax.set_ylabel("Predicted (kWh)")
    ax.set_xlim(0,lim); ax.set_ylim(0,lim)
plt.tight_layout()
plt.savefig("grafico_B_scatter.png",dpi=150,bbox_inches="tight")
plt.show(); print("  ✓ Chart B")

# ── Chart C: feature importance ───────────────────────────────────────────────
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9,7))
fig.suptitle("C  –  Feature importance", fontsize=13, fontweight="bold")
col_bars = ["#993C1D" if "lag" in f else
            "#BA7517" if "produzione" in f else
            "#185FA5" if any(x in f for x in ["ora","giorno","mese","weekend","festivo"]) else
            "#0F6E56" for f in imp.index]
bars = ax.barh(imp.index, imp.values, color=col_bars, edgecolor="white", height=0.7)
for bar, val in zip(bars, imp.values):
    ax.text(val+0.001, bar.get_y()+bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#993C1D", label="Lag features (historical memory)"),
                   Patch(color="#185FA5", label="Temporal features"),
                   Patch(color="#BA7517", label="PV production"),
                   Patch(color="#0F6E56", label="User type")],
          fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig("grafico_C_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show(); print("  ✓ Chart C")

# ── Chart D: shared energy · Mon–Wed · October ───────────────────────────────
tm = test[test["mese"]==10].copy()
tm["ei_r"] = np.maximum(tm["produzione_fv_kwh"]-tm["consumo_kwh"], 0)
tm["cr_r"] = np.maximum(tm["consumo_kwh"]-tm["produzione_fv_kwh"], 0)
tm["ei_p"] = np.maximum(tm["produzione_fv_kwh"]-tm["consumo_previsto_kwh"], 0)
tm["cr_p"] = np.maximum(tm["consumo_previsto_kwh"]-tm["produzione_fv_kwh"], 0)

pa = tm.groupby("timestamp").agg(
    consumo_r=("consumo_kwh","sum"), consumo_p=("consumo_previsto_kwh","sum"),
    fv=("produzione_fv_kwh","sum"),
    ei_r=("ei_r","sum"), cr_r=("cr_r","sum"),
    ei_p=("ei_p","sum"), cr_p=("cr_p","sum"),
).reset_index().sort_values("timestamp").reset_index(drop=True)
pa["ec_r"] = np.minimum(pa["cr_r"], pa["ei_r"])
pa["ec_p"] = np.minimum(pa["cr_p"], pa["ei_p"])
pa["dow"]  = pa["timestamp"].dt.dayofweek

# Find first Monday (dayofweek == 0), take 72-hour window Mon–Wed
lun_idx = pa[pa["dow"]==0].index[0]
pa3 = pa.iloc[lun_idx : lun_idx+72].reset_index(drop=True)

fig, axes = plt.subplots(2,1,figsize=(10,9))
fig.suptitle("D  –  CER energy analysis · Mon–Wed · October 2023",
             fontsize=13, fontweight="bold")
for ax, (cons, ec, col, tit) in zip(axes, [
    ("consumo_r","ec_r","#0F6E56","Actual data"),
    ("consumo_p","ec_p","#993C1D","Model predictions"),
]):
    x = range(len(pa3))
    ax.fill_between(x, pa3["fv"],   alpha=0.25, color=COLORS["fv"],    label="PV production")
    ax.fill_between(x, pa3[cons],   alpha=0.20, color=COLORS["reale"], label="CER consumption")
    ax.fill_between(x, pa3[ec],     alpha=0.55, color=col,             label="Shared energy")
    ax.plot(x, pa3["fv"],  color=COLORS["fv"],   lw=1.5)
    ax.plot(x, pa3[cons],  color=COLORS["reale"], lw=1.5)
    ax.set_title(tit); ax.set_ylabel("kWh"); ax.legend(fontsize=9)
    ax.set_xticks([d*24+12 for d in range(3)])
    ax.set_xticklabels(["Mon","Tue","Wed"], fontsize=10)
    for d in range(1,3): ax.axvline(d*24, color="gray", lw=0.4, alpha=0.4)
plt.tight_layout()
plt.savefig("grafico_D_energia_condivisa.png", dpi=150, bbox_inches="tight")
plt.show(); print("  ✓ Chart D")

# ── Chart E: PV surplus vs deficit ───────────────────────────────────────────
at = test.copy()
at["ei"] = np.maximum(at["produzione_fv_kwh"]-at["consumo_kwh"],0)
at["cr"] = np.maximum(at["consumo_kwh"]-at["produzione_fv_kwh"],0)
at["ei_p"] = np.maximum(at["produzione_fv_kwh"]-at["consumo_previsto_kwh"],0)
at["cr_p"] = np.maximum(at["consumo_previsto_kwh"]-at["produzione_fv_kwh"],0)
at2 = at.groupby("timestamp").agg(
    ei=("ei","sum"),cr=("cr","sum"),
    ei_p=("ei_p","sum"),cr_p=("cr_p","sum")).reset_index()
at2["regime"] = np.where(at2["ei"]>at2["cr"],"surplus","deficit")
at2["ec_r"]   = np.minimum(at2["cr"],at2["ei"])
at2["ec_p"]   = np.minimum(at2["cr_p"],at2["ei_p"])
at2["err_ec"] = np.abs(at2["ec_r"]-at2["ec_p"])
at2["mese"]   = at2["timestamp"].dt.month

ore_reg = at2.groupby(["mese","regime"]).size().unstack(fill_value=0)
mp=ore_reg.index.tolist(); x=np.arange(len(mp)); w=0.4
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle("E  –  PV surplus vs deficit · test months",
             fontsize=13,fontweight="bold")
axes[0].bar(x-w/2,ore_reg.get("surplus",pd.Series(0,index=ore_reg.index)),w,
            color=COLORS["fv"],alpha=0.85,label="Surplus")
axes[0].bar(x+w/2,ore_reg.get("deficit",pd.Series(0,index=ore_reg.index)),w,
            color=COLORS["neutral"],alpha=0.55,label="Deficit")
axes[0].set_xticks(x); axes[0].set_xticklabels([MESI[m-1] for m in mp])
axes[0].set_ylabel("Hours"); axes[0].legend()
axes[0].set_title("Hours by energy regime")
sur=at2[at2["regime"]=="surplus"]; dft=at2[at2["regime"]=="deficit"]
ms_v=mean_absolute_error(sur["ec_r"],sur["ec_p"]) if len(sur) else 0
md_v=mean_absolute_error(dft["ec_r"],dft["ec_p"]) if len(dft) else 0
bars2=axes[1].bar(["Surplus\n(ML-driven)","Deficit\n(PV-driven)"],
                  [ms_v,md_v],color=[COLORS["fv"],COLORS["neutral"]],
                  alpha=0.85,width=0.4)
for b,v in zip(bars2,[ms_v,md_v]):
    axes[1].text(b.get_x()+b.get_width()/2,v+0.001,f"{v:.4f}",
                 ha="center",fontweight="bold")
axes[1].set_ylabel("Shared energy MAE (kWh)")
axes[1].set_title("Shared energy error by regime")
plt.tight_layout()
plt.savefig("grafico_E_surplus_deficit.png",dpi=150,bbox_inches="tight")
plt.show(); print("  ✓ Chart E")


# ══════════════════════════════════════════════════════════════════════════════
# PART 4 – ECONOMIC IMPACT
# ══════════════════════════════════════════════════════════════════════════════

print("\n▶ PART 4 – Economic impact")

TARIFFA     = 0.130   # GSE base incentive rate (€/kWh)
TARIFFA_MIN = 0.100
TARIFFA_MAX = 0.130

at2["errore_kwh"]  = at2["err_ec"]
at2["errore_euro"] = at2["errore_kwh"] * TARIFFA

per_mese = at2[at2["regime"]=="surplus"].groupby("mese").agg(
    ore_surplus         =("errore_kwh","count"),
    errore_surplus_kwh  =("errore_kwh","sum"),
    errore_surplus_euro =("errore_euro","sum"),
).reset_index()

errore_test    = per_mese["errore_surplus_euro"].sum()
errore_annuo   = errore_test * 3
errore_ann_min = per_mese["errore_surplus_kwh"].sum()*3*TARIFFA_MIN
errore_ann_max = per_mese["errore_surplus_kwh"].sum()*3*TARIFFA_MAX

print(f"  Economic error on test set (Sep–Dec): {errore_test:.2f} €")
print(f"  Annual projection (base rate):        {errore_annuo:.2f} €/year")
print(f"  Rate sensitivity range (min–max):     {errore_ann_min:.1f}–{errore_ann_max:.1f} €/year")

fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle("F  –  Economic impact of ML errors · GSE incentives",
             fontsize=13,fontweight="bold")
ml=[MESI[m-1] for m in per_mese["mese"]]
axes[0].bar(ml,per_mese["errore_surplus_euro"],
            color=COLORS["previsto"],alpha=0.85,edgecolor="white",width=0.5)
axes[0].set_ylabel("€/month"); axes[0].set_title("Monthly economic error (100 €/MWh)")
for i,v in enumerate(per_mese["errore_surplus_euro"]):
    axes[0].text(i,v+0.01,f"{v:.2f}€",ha="center",fontsize=9,fontweight="bold")
tariffe=[60,70,80,90,100,110,120,130]
errori_a=[per_mese["errore_surplus_kwh"].sum()*3*t/1000 for t in tariffe]
col_t=["#B5D4F4" if t!=130 else "#185FA5" for t in tariffe]
axes[1].bar([str(t) for t in tariffe],errori_a,color=col_t,edgecolor="white",width=0.6)
axes[1].set_xlabel("GSE rate (€/MWh)"); axes[1].set_ylabel("€/year")
axes[1].set_title("Rate sensitivity – annual projection")
for b,v in zip(axes[1].patches,errori_a):
    axes[1].text(b.get_x()+b.get_width()/2,v+0.2,f"{v:.1f}",ha="center",fontsize=8)
plt.tight_layout()
plt.savefig("grafico_F_impatto_economico.png",dpi=150,bbox_inches="tight")
plt.show(); print("  ✓ Chart F")

per_mese.to_csv("cer_impatto_economico.csv",index=False)

# ── Create download ZIP ───────────────────────────────────────────────────────
print("\n▶ Creating ZIP archive ...")

file_list = [
    "cer_dataset_completo.csv",
    "cer_energia_condivisa.csv",
    "cer_previsioni.csv",
    "cer_impatto_economico.csv",
    "grafico_A_reale_vs_previsto.png",
    "grafico_B_scatter.png",
    "grafico_C_feature_importance.png",
    "grafico_D_energia_condivisa.png",
    "grafico_E_surplus_deficit.png",
    "grafico_F_impatto_economico.png",
]

with zipfile.ZipFile("CER_tesi.zip","w") as z:
    for f in file_list:
        if os.path.exists(f):
            z.write(f); print(f"  + {f}")
        else:
            print(f"  ✗ missing: {f}")

with open("CER_tesi.zip","rb") as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(
    '<a download="CER_tesi.zip" '
    'href="data:application/zip;base64,' + b64 + '" '
    'style="display:inline-block;padding:12px 24px;background:#185FA5;'
    'color:white;border-radius:8px;text-decoration:none;'
    'font-size:14px;font-weight:500;">Download CER_tesi.zip</a>'
))
print("\n✓ Pipeline complete.")